# 00 Prepare External Phonons

This notebook uses the local dftbephy Graphene example as an external phonon source and converts selected q-points into DeePTB `Phonons` NPZ files. The electronic model is created directly from matsci-0-3 SKF files through DeePTB `DFTBSK`; no `.pth` checkpoint is required.

In [ ]:
from pathlib import Path
import json
import numpy as np

from dptb.nn.dftbsk import DFTBSK
from dptb.postprocess.unified import TBSystem
from dptb.postprocess.unified.eph import Phonons
from epc_notebook_utils import external_path, regularize_tiny_negative_frequencies

ROOT = Path.cwd()
WORK = ROOT / "work"
WORK.mkdir(exist_ok=True)

# Set these environment variables before running the notebooks.
DFTBEPHY_GRAPHENE = external_path(
    "DEEPTB_EPC_DFTBEPHY_GRAPHENE",
    "the dftbephy Graphene example directory, e.g. /path/to/dftbephy/examples/Graphene",
)
MATSCI_SK = external_path(
    "DEEPTB_EPC_MATSCI_SK",
    "the matsci-0-3 SKF directory containing C-C.skf",
)

STRUCTURE = ROOT / "graphene.vasp"
if not STRUCTURE.exists():
    STRUCTURE = DFTBEPHY_GRAPHENE / "opt" / "geo_end.gen"

PHONOPY_YAML = DFTBEPHY_GRAPHENE / "phonons" / "phonopy_disp.yaml"
FORCE_SETS = DFTBEPHY_GRAPHENE / "phonons" / "FORCE_SETS"

# matsci-0-3 C-C.skf contains the carbon SK data used by this example model.
BASIS = {"C": ["2s", "2p"]}

for path in [MATSCI_SK, STRUCTURE, PHONOPY_YAML, FORCE_SETS]:
    if not path.exists():
        raise FileNotFoundError(path)

model = DFTBSK(
    basis=BASIS,
    skdata=str(MATSCI_SK),
    overlap=True,
    dtype="float64",
    device="cpu",
    r_max={'C': 6.349479778742587},
    interp_method="smooth_intp"
)

model.eval()

system = TBSystem(data=str(STRUCTURE), calculator=model)

print(system.atoms)
print(system.model.name, system.model.basis)


## Select q-points and electronic k-points

The q-points below are intentionally small for an example notebook. For production, replace them with a converged path or mesh.

In [ ]:
q_path = np.array([
    [0.0, 0.0, 0.0],
    [1.0 / 6.0, 1.0 / 6.0, 0.0],
    [1.0 / 3.0, 1.0 / 3.0, 0.0],
    [0.5, 0.0, 0.0],
    [0.0, 0.0, 0.0],
], dtype=float)

q_mesh = np.array([
    [0.0, 0.0, 0.0],
    [0.5, 0.0, 0.0],
    [0.0, 0.5, 0.0],
    [0.5, 0.5, 0.0],
], dtype=float)

fixed_kpoints = np.array([[0.0, 0.0, 0.0]], dtype=float)
np.save(WORK / "q_path.npy", q_path)
np.save(WORK / "q_mesh.npy", q_mesh)
(WORK / "fixed_kpoints.json").write_text(json.dumps({"kpoints": fixed_kpoints.tolist()}, indent=2))
print(WORK)


## Convert phonopy output

`Phonons.from_phonopy_file(...)` calls phonopy as a parser. DeePTB still does not calculate phonons or force constants.

In [ ]:
phonons_path = Phonons.from_phonopy_file(
    PHONOPY_YAML,
    qpoints=q_path,
    force_sets_filename=str(FORCE_SETS),
)

phonons_path = regularize_tiny_negative_frequencies(phonons_path)
phonons_path.save_npz(WORK / "phonons_qpath.npz")

phonons_mesh = Phonons.from_phonopy_file(
    PHONOPY_YAML,
    qpoints=q_mesh,
    force_sets_filename=str(FORCE_SETS),
)

phonons_mesh = regularize_tiny_negative_frequencies(phonons_mesh)
phonons_mesh.save_npz(WORK / "phonons_mesh.npz")

print("q-path phonons:", phonons_path.qpoints.shape, phonons_path.frequencies.shape)
print("mesh phonons:", phonons_mesh.qpoints.shape, phonons_mesh.frequencies.shape)
print("metadata:", phonons_mesh.metadata)



## Quick electronic sanity check

In [ ]:
kpoints = np.array([[0.0, 0.0, 0.0], [1/3, 1/3, 0.0]], dtype=float)
_, evals = system.get_eigenvalues(k_points=kpoints)
evals = np.asarray(evals.detach().cpu() if hasattr(evals, "detach") else evals)

print(evals.shape)
print(evals[:, :8])